# Real Image Editing: DDIM Inversion + Null-text Optimization

This notebook runs the full inversion pipeline on Google Colab (A100).

**Steps:**
1. Install dependencies & clone repo
2. Login to HuggingFace (needed for SD 2.1)
3. Upload a test image
4. Run DDIM inversion + null-text optimization
5. View reconstruction results

## 1. Setup

In [ ]:
# Clone repo and install dependencies
!git clone https://github.com/beratcelik1/real-image-editing.git
%cd real-image-editing
!pip install -q torch diffusers transformers accelerate safetensors Pillow numpy torchvision scikit-image tqdm matplotlib

In [ ]:
# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# HuggingFace login via Colab Secrets (no widget needed)
# To set up: click the key icon in the left sidebar > add secret named "HF_TOKEN" with your token
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in via Colab secret")
except Exception:
    import os
    if os.environ.get("HF_TOKEN"):
        login(token=os.environ["HF_TOKEN"])
        print("Logged in via env var")
    else:
        print("No HF token found. Add 'HF_TOKEN' in Colab secrets (key icon, left sidebar).")

In [ ]:
# Option B: Download a sample image (skip if you uploaded one above)
# !wget -O data/sample.jpg "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg"
# IMAGE_PATH = "data/sample.jpg"

# Download sample test image
!wget -q -O data/sample.jpg "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=512"
IMAGE_PATH = "data/sample.jpg"
PROMPT = "a photo of a cat looking at the camera"
print(f"Image: {IMAGE_PATH}")

In [ ]:
# === CONFIGURE THESE ===
PROMPT = "a photo of a cat sitting on a couch"  # <-- describe your image
NUM_STEPS = 50
CFG_SCALE = 7.5
OPT_STEPS = 10  # null-text optimization steps per timestep
LR = 1e-2
# =======================

In [ ]:
# Load and encode image
image = load_image(IMAGE_PATH)
latent = encode_image(image, vae, device)
print(f"Image size: {image.size}, Latent shape: {latent.shape}")

# Display original
from IPython.display import display
display(image)

In [ ]:
# Get text embeddings
text_emb = get_text_embeddings(PROMPT, tokenizer, text_encoder, device)
uncond_emb = get_uncond_embeddings(tokenizer, text_encoder, device)
print(f"Text embedding shape: {text_emb.shape}")
print(f"Uncond embedding shape: {uncond_emb.shape}")

In [ ]:
# Hyperparameters
NUM_STEPS = 50
CFG_SCALE = 7.5
OPT_STEPS = 10
LR = 1e-2

In [ ]:
# Stage 2: Null-text Optimization (~5 min on A100)
print("Running null-text optimization...")
null_embeddings = null_text_optimization(
    latents_trajectory, text_emb, uncond_emb, unet, scheduler,
    num_steps=NUM_STEPS, cfg_scale=CFG_SCALE,
    opt_steps=OPT_STEPS, lr=LR,
)
print(f"Optimized {len(null_embeddings)} null embeddings")

In [ ]:
# Stage 3: Reconstruction (~1 min on A100)
print("Reconstructing...")
recon_latent = reconstruct(
    latent_T, null_embeddings, text_emb, unet, scheduler,
    num_steps=NUM_STEPS, cfg_scale=CFG_SCALE,
)
recon_image = decode_latent(recon_latent, vae)
print("Done!")

## 5. Results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# Compute metrics
orig_np = np.array(image).astype(np.float32)
recon_np = np.array(recon_image).astype(np.float32)
psnr = peak_signal_noise_ratio(orig_np, recon_np, data_range=255.0)
ssim = structural_similarity(orig_np, recon_np, channel_axis=2, data_range=255.0)

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(image)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(recon_image)
axes[1].set_title(f"Reconstruction (PSNR={psnr:.1f}dB, SSIM={ssim:.3f})")
axes[1].axis("off")
plt.tight_layout()
plt.savefig("outputs/reconstruction_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nPSNR: {psnr:.2f} dB")
print(f"SSIM: {ssim:.4f}")
print(f"\nTarget: PSNR > 30 dB, SSIM > 0.9")

In [ ]:
# Pixel-level difference heatmap
diff = np.abs(orig_np - recon_np).mean(axis=2)

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
im = ax.imshow(diff, cmap="hot")
ax.set_title("Absolute Pixel Difference (mean across channels)")
ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.savefig("outputs/diff_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Save outputs
from pathlib import Path
from PIL import Image as PILImage

Path("outputs").mkdir(exist_ok=True)
image.save("outputs/original.png")
recon_image.save("outputs/reconstruction.png")

# Save null embeddings for later use by teammates (P2P/PnP editing)
torch.save({
    "null_embeddings": null_embeddings,
    "latent_T": latent_T,
    "latents_trajectory": latents_trajectory,
    "text_embeddings": text_emb,
    "prompt": PROMPT,
    "num_steps": NUM_STEPS,
    "cfg_scale": CFG_SCALE,
}, "outputs/inversion_cache.pt")

print("Saved: outputs/original.png, outputs/reconstruction.png, outputs/inversion_cache.pt")
print("\ninversion_cache.pt contains everything your teammates need for P2P/PnP editing.")

## 6. Batch Test (Multiple Images)

Run inversion on multiple images to verify robustness.

In [ ]:
# Batch test on multiple images (optional)
# Add image paths and prompts here:
test_cases = [
    # ("path/to/image1.jpg", "a photo of ..."),
    # ("path/to/image2.jpg", "a photo of ..."),
]

results = []
for img_path, prompt in test_cases:
    img = load_image(img_path)
    lat = encode_image(img, vae, device)
    t_emb = get_text_embeddings(prompt, tokenizer, text_encoder, device)
    u_emb = get_uncond_embeddings(tokenizer, text_encoder, device)

    lat_T, traj = ddim_inversion(lat, t_emb, u_emb, unet, scheduler)
    null_embs = null_text_optimization(traj, t_emb, u_emb, unet, scheduler)
    rec_lat = reconstruct(lat_T, null_embs, t_emb, unet, scheduler)
    rec_img = decode_latent(rec_lat, vae)

    o = np.array(img).astype(np.float32)
    r = np.array(rec_img).astype(np.float32)
    p = peak_signal_noise_ratio(o, r, data_range=255.0)
    s = structural_similarity(o, r, channel_axis=2, data_range=255.0)
    results.append({"image": img_path, "psnr": p, "ssim": s})
    print(f"{img_path}: PSNR={p:.1f}dB, SSIM={s:.3f}")

if results:
    avg_psnr = np.mean([r["psnr"] for r in results])
    avg_ssim = np.mean([r["ssim"] for r in results])
    print(f"\nAverage: PSNR={avg_psnr:.1f}dB, SSIM={avg_ssim:.3f}")